# stock_return

> 对应代码：`EconometricsCode/code_in_notes/Chap.25/stock_return.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.25`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.25")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们载入日度股票行情数据，先整理为月度数据：从日期中提取年、月、日，用 egen 配合 by(year month) 找出每个月的最大交易日，只保留每月最后一个交易日的收盘价作为月末价格；再用 ym() 函数把年月合成为 Stata 的月度日期变量，并设为 %tm 显示格式。

In [ ]:
%%stata
use ../datasets/stock_price, clear
// 整理为月度数据
gen year=year(date)
gen month=month(date)
gen day=day(date)
egen max_day=max(day), by(year month)
keep if day==max_day
gen month_date=ym(year,month)
format month_date %tm

接着把数据声明为以 stkcd 为个体、month_date 为时间的面板，并计算两种月度收益率：简单收益率 R_t = (P_t − P_{t−1})/P_{t−1} 直接刻画投资收益的百分比；对数收益率 r_t = ln P_t − ln P_{t−1} 具有跨期可加性（多期对数收益率等于单期之和），在收益率较小时二者近似相等。加上标签以便图例显示。

In [ ]:
%%stata
// 设定时间并计算收益率
tsset stkcd month_date
gen simple_rr=(clsprc-L.clsprc)/L.clsprc
label variable simple_rr "简单收益率"
gen log_rr=log(clsprc)-log(L.clsprc)
label variable log_rr "对数收益率"

最后我们画出 600519 这只股票的两种月度收益率曲线，实线为简单收益率、虚线为对数收益率，可以看到二者几乎重合，验证了月度频率下两种收益率口径的近似等价。

In [ ]:
%%stata
tsline simple_rr log_rr if stkcd == 600519, ///
	lpattern(solid dash)